# Telco Genie Learning Day — UC documentation & constraints
Run **after** `01_generate_dataset.py`. Adds **table/column comments** and informational **PRIMARY KEY** / **FOREIGN KEY** constraints so Genie (and other tools) can use semantics and join relationships.

| Step | What |
|------|------|
| 1 | `COMMENT ON TABLE` / `COMMENT ON COLUMN` |
| 2 | `PRIMARY KEY` on each table (order respects FK dependencies) |
| 3 | `FOREIGN KEY` from fact tables to dimensions |

Constraints are [informational](https://docs.databricks.com/aws/en/tables/constraints) (not engine-enforced); orphan rows will not fail loads.

## Configuration
Must match `01_generate_dataset.py`.

In [0]:
CATALOG = "workspace"
SCHEMA = "telco"

spark.sql(f"USE {CATALOG}.{SCHEMA}")
print(f"Target: {CATALOG}.{SCHEMA}")

## 1. Table & column comments

In [0]:
def q(sql: str) -> None:
    spark.sql(sql)


def sql_lit(s: str) -> str:
    """Single-quoted SQL string literal (escape apostrophes)."""
    return "'" + s.replace("'", "''") + "'"

# --- plans ---
q(f"""
COMMENT ON TABLE {CATALOG}.{SCHEMA}.plans IS
  'Reference dimension: mobile plan tiers (Essential through Premium). One row per plan_id.';
""")
for col, text in [
    ("plan_id", "Stable plan identifier (e.g. BAS-30). Join key to customers.plan_id."),
    ("plan_name", "Marketing name for the plan tier."),
    ("plan_category", "Prepaid vs Postpaid."),
    ("monthly_cost_aud", "Recurring monthly charge in Australian dollars (AUD)."),
    ("data_allowance_gb", "Included monthly data in GB; 9999 means unlimited."),
    ("included_calls_min", "Included national call minutes (simplified cap)."),
]:
    q(f"COMMENT ON COLUMN {CATALOG}.{SCHEMA}.plans.{col} IS {sql_lit(text)};")

# --- customers ---
q(f"""
COMMENT ON TABLE {CATALOG}.{SCHEMA}.customers IS
  'Subscriber dimension: ~10K customers with location, plan, device, and churn status. Grain: one row per customer_id.';
""")
for col, text in [
    ("customer_id", "Unique subscriber identifier (CUST-000001 format). Primary person key."),
    ("state", "Australian state or territory code."),
    ("city", "Service / billing city."),
    ("region_type", "Metro vs Regional — used in churn and CX analysis."),
    ("signup_date", "Account creation date."),
    ("plan_id", "FK to plans.plan_id — current or last plan."),
    ("device_type", "Handset network capability (4G vs 5G)."),
    ("status", "Active or Churned at end of observation window."),
    ("churn_date", "Date churn observed, if status = Churned; else null."),
]:
    q(f"COMMENT ON COLUMN {CATALOG}.{SCHEMA}.customers.{col} IS {sql_lit(text)};")

# --- usage ---
q(f"""
COMMENT ON TABLE {CATALOG}.{SCHEMA}.usage IS
  'Monthly usage facts: data, calls, SMS, overage per customer per month (Jan–Mar 2026).';
""")
for col, text in [
    ("customer_id", "FK to customers.customer_id."),
    ("month", "Calendar month as YYYY-MM (e.g. 2026-01)."),
    ("data_used_gb", "Total mobile data consumed in the month (GB)."),
    ("calls_min", "Total voice minutes used."),
    ("sms_count", "Total SMS messages sent."),
    ("overage_charges_aud", "Extra charges in AUD when usage exceeds allowance."),
]:
    q(f"COMMENT ON COLUMN {CATALOG}.{SCHEMA}.usage.{col} IS {sql_lit(text)};")

# --- support_tickets ---
q(f"""
COMMENT ON TABLE {CATALOG}.{SCHEMA}.support_tickets IS
  'CX facts: support interactions, NPS, resolution time. Grain: one row per ticket_id.';
""")
for col, text in [
    ("ticket_id", "Unique ticket identifier (TKT-000001 format)."),
    ("customer_id", "FK to customers.customer_id."),
    ("created_date", "Ticket opened date."),
    ("category", "High-level issue category (Billing, Network, etc.)."),
    ("subcategory", "Specific issue type within category."),
    ("severity", "Low / Medium / High / Critical."),
    ("channel", "Contact channel (App, Phone, etc.)."),
    ("resolution_time_hours", "Hours to resolve or close."),
    ("nps_score", "Net Promoter-style score 0–10 after interaction."),
]:
    q(f"COMMENT ON COLUMN {CATALOG}.{SCHEMA}.support_tickets.{col} IS {sql_lit(text)};")

# --- network_events ---
q(f"""
COMMENT ON TABLE {CATALOG}.{SCHEMA}.network_events IS
  'Network operations: outages and maintenance by location. Not tied to individual customers in this synthetic set.';
""")
for col, text in [
    ("event_id", "Unique event identifier (EVT-0001 format)."),
    ("event_date", "Date of the event."),
    ("state", "Australian state or territory."),
    ("city", "City where impact was centred."),
    ("region_type", "Metro vs Regional."),
    ("event_type", "Planned Maintenance, Unplanned Outage, etc."),
    ("duration_min", "Estimated or actual duration in minutes."),
    ("affected_customers", "Modelled count of impacted subscribers."),
]:
    q(f"COMMENT ON COLUMN {CATALOG}.{SCHEMA}.network_events.{col} IS {sql_lit(text)};")

print("Comments applied.")

## 2. Primary keys
Tables created with `saveAsTable` default to **nullable** columns. [Primary keys](https://docs.databricks.com/aws/en/tables/constraints) require those columns to be **NOT NULL** — set that first, then add constraints. Order: drop existing constraints → `ALTER COLUMN … SET NOT NULL` on PK/FK columns → add PKs → add FKs.

In [0]:
# Drop FKs first, then PKs (safe re-run). IF EXISTS avoids errors on first run.
for stmt in [
    f"ALTER TABLE {CATALOG}.{SCHEMA}.support_tickets DROP CONSTRAINT IF EXISTS fk_support_tickets_customer",
    f"ALTER TABLE {CATALOG}.{SCHEMA}.usage DROP CONSTRAINT IF EXISTS fk_usage_customer",
    f"ALTER TABLE {CATALOG}.{SCHEMA}.customers DROP CONSTRAINT IF EXISTS fk_customers_plan",
    f"ALTER TABLE {CATALOG}.{SCHEMA}.support_tickets DROP CONSTRAINT IF EXISTS pk_support_tickets",
    f"ALTER TABLE {CATALOG}.{SCHEMA}.usage DROP CONSTRAINT IF EXISTS pk_usage",
    f"ALTER TABLE {CATALOG}.{SCHEMA}.customers DROP CONSTRAINT IF EXISTS pk_customers",
    f"ALTER TABLE {CATALOG}.{SCHEMA}.plans DROP CONSTRAINT IF EXISTS pk_plans",
    f"ALTER TABLE {CATALOG}.{SCHEMA}.network_events DROP CONSTRAINT IF EXISTS pk_network_events",
]:
    spark.sql(stmt)

# PK/FK columns must be NOT NULL before adding named constraints
for table, col in [
    (f"{CATALOG}.{SCHEMA}.plans", "plan_id"),
    (f"{CATALOG}.{SCHEMA}.customers", "customer_id"),
    (f"{CATALOG}.{SCHEMA}.customers", "plan_id"),
    (f"{CATALOG}.{SCHEMA}.usage", "customer_id"),
    (f"{CATALOG}.{SCHEMA}.usage", "month"),
    (f"{CATALOG}.{SCHEMA}.support_tickets", "ticket_id"),
    (f"{CATALOG}.{SCHEMA}.support_tickets", "customer_id"),
    (f"{CATALOG}.{SCHEMA}.network_events", "event_id"),
]:
    q(f"ALTER TABLE {table} ALTER COLUMN {col} SET NOT NULL;")

print("Key columns set NOT NULL.")

q(f"ALTER TABLE {CATALOG}.{SCHEMA}.plans ADD CONSTRAINT pk_plans PRIMARY KEY (plan_id);")
q(f"ALTER TABLE {CATALOG}.{SCHEMA}.customers ADD CONSTRAINT pk_customers PRIMARY KEY (customer_id);")
q(f"ALTER TABLE {CATALOG}.{SCHEMA}.usage ADD CONSTRAINT pk_usage PRIMARY KEY (customer_id, month);")
q(f"ALTER TABLE {CATALOG}.{SCHEMA}.support_tickets ADD CONSTRAINT pk_support_tickets PRIMARY KEY (ticket_id);")
q(f"ALTER TABLE {CATALOG}.{SCHEMA}.network_events ADD CONSTRAINT pk_network_events PRIMARY KEY (event_id);")

print("Primary keys added.")

## 3. Foreign keys

In [0]:
q(f"""
ALTER TABLE {CATALOG}.{SCHEMA}.customers
  ADD CONSTRAINT fk_customers_plan
  FOREIGN KEY (plan_id) REFERENCES {CATALOG}.{SCHEMA}.plans (plan_id);
""")
q(f"""
ALTER TABLE {CATALOG}.{SCHEMA}.usage
  ADD CONSTRAINT fk_usage_customer
  FOREIGN KEY (customer_id) REFERENCES {CATALOG}.{SCHEMA}.customers (customer_id);
""")
q(f"""
ALTER TABLE {CATALOG}.{SCHEMA}.support_tickets
  ADD CONSTRAINT fk_support_tickets_customer
  FOREIGN KEY (customer_id) REFERENCES {CATALOG}.{SCHEMA}.customers (customer_id);
""")

print("Foreign keys added.")

## 4. Verify (`information_schema`)

In [0]:
%sql
SELECT tc.table_name, tc.constraint_type, tc.constraint_name
FROM workspace.information_schema.table_constraints tc
WHERE tc.table_catalog = 'workspace'
  AND tc.table_schema = 'telco'
  AND tc.constraint_type IN ('PRIMARY KEY', 'FOREIGN KEY')
ORDER BY tc.table_name, tc.constraint_type, tc.constraint_name

## Done
**Next:** Create Metric View(s), then configure the Genie space on this catalog/schema.